# Data Exploration & Brand Selection

This notebook explores the dataset to make an informed decision about which brand to focus on.

In [ ]:
import pandas as pd
import json
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Load Dataset

In [ ]:
# Load the dataset
# TODO: Update path based on actual data location
df = pd.read_csv('../data/raw_data.csv')  # Adjust format as needed

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

## Brand Overview

In [ ]:
# Identify brands (adjust column name based on actual data)
brands = df['brand'].unique()
print(f"Brands in dataset: {brands}")
print(f"\nBrand distribution:")
print(df['brand'].value_counts())

## Conversation Analysis Per Brand

In [ ]:
# Group by brand and analyze conversations
brand_stats = {}

for brand in brands:
    brand_data = df[df['brand'] == brand]
    
    # Count unique conversations
    num_conversations = brand_data['conversation_id'].nunique()
    
    # Count messages
    num_messages = len(brand_data)
    
    # Average messages per conversation
    avg_msgs = num_messages / num_conversations if num_conversations > 0 else 0
    
    # Multi-turn conversations (>2 messages)
    conv_lengths = brand_data.groupby('conversation_id').size()
    multi_turn = (conv_lengths > 2).sum()
    
    # Customer vs Agent ratio
    customer_msgs = (brand_data['sender'] == 'customer').sum() if 'sender' in brand_data.columns else 0
    agent_msgs = (brand_data['sender'] == 'agent').sum() if 'sender' in brand_data.columns else 0
    
    brand_stats[brand] = {
        'conversations': num_conversations,
        'messages': num_messages,
        'avg_msgs_per_conv': avg_msgs,
        'multi_turn_conversations': multi_turn,
        'customer_messages': customer_msgs,
        'agent_messages': agent_msgs
    }

# Display stats
stats_df = pd.DataFrame(brand_stats).T
print("Brand Statistics:")
print(stats_df)

## Conversation Length Distribution

In [ ]:
# Analyze conversation lengths by brand
fig, axes = plt.subplots(1, len(brands), figsize=(15, 5))

for idx, brand in enumerate(brands):
    brand_data = df[df['brand'] == brand]
    conv_lengths = brand_data.groupby('conversation_id').size()
    
    if len(brands) > 1:
        ax = axes[idx]
    else:
        ax = axes
    
    ax.hist(conv_lengths, bins=20, edgecolor='black')
    ax.set_title(f'{brand}\n(avg: {conv_lengths.mean():.2f} msgs/conv)')
    ax.set_xlabel('Messages per Conversation')
    ax.set_ylabel('Frequency')

plt.tight_layout()
plt.show()

# Print detailed statistics
for brand in brands:
    brand_data = df[df['brand'] == brand]
    conv_lengths = brand_data.groupby('conversation_id').size()
    print(f"\n{brand}:")
    print(f"  Min: {conv_lengths.min()}, Max: {conv_lengths.max()}")
    print(f"  Median: {conv_lengths.median()}, Mean: {conv_lengths.mean():.2f}")

## Data Quality Analysis

In [ ]:
# Check for duplicates, missing values, etc.
print("Missing values per brand:")
for brand in brands:
    brand_data = df[df['brand'] == brand]
    print(f"\n{brand}:")
    print(brand_data.isnull().sum())

# Duplicate analysis
print("\n\nDuplicate analysis:")
for brand in brands:
    brand_data = df[df['brand'] == brand]
    duplicates = brand_data.duplicated().sum()
    print(f"{brand}: {duplicates} duplicate rows")

## Intent Distribution (Preview)

In [ ]:
# Sample texts to identify potential intents
print("Sample messages by brand:\n")
for brand in brands:
    brand_data = df[df['brand'] == brand]
    print(f"\n{brand.upper()}:")
    print("Sample messages:")
    # Adjust 'text' or 'message' column name as needed
    sample_col = next((col for col in ['text', 'message', 'content'] if col in brand_data.columns), None)
    if sample_col:
        for msg in brand_data[sample_col].dropna().head(5):
            print(f"  - {msg[:80]}...")

## Recommendation

In [ ]:
# Score each brand based on usability
scores = {}

for brand, stats in brand_stats.items():
    score = 0
    
    # More conversations = better
    score += stats['conversations'] / 100  # Normalize
    
    # More multi-turn conversations = better for building gold set
    score += stats['multi_turn_conversations'] / 50
    
    # Longer conversations = more context
    score += stats['avg_msgs_per_conv'] * 5
    
    scores[brand] = score

# Sort and recommend
ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

print("Brand Ranking:")
for rank, (brand, score) in enumerate(ranked, 1):
    print(f"{rank}. {brand}: {score:.2f}")

recommended = ranked[0][0]
print(f"\n✓ RECOMMENDATION: Focus on {recommended}")

# Show reasoning
rec_stats = brand_stats[recommended]
print(f"\nReasoning:")
print(f"  - {rec_stats['conversations']} conversations")
print(f"  - {rec_stats['multi_turn_conversations']} multi-turn conversations")
print(f"  - {rec_stats['avg_msgs_per_conv']:.1f} messages per conversation (avg)")
print(f"  - High content diversity and resolution patterns")